In [53]:
import pandas as pd

data = pd.read_csv("/Users/mashen/mashitup_mixes/data.csv")

data["songs"] = data.groupby("mix")["track"].transform('max')
data["position"] = data["track"] / data["songs"]
print(data)

           mix  track  key         bpm     loudness       rms         energy  \
0    yousuke_1      0   1A  125.454910   728.642578  0.097136   18724.476562   
1    yousuke_1      1   7B  152.852615  2717.464355  0.156905  133541.906250   
2    yousuke_1      2   5B  152.921234  2715.843750  0.165844  133423.062500   
3    yousuke_1      3   4A  149.566284  3713.772217  0.144237  212854.265625   
4    yousuke_1      4   5A  146.313080  3994.437988  0.133264  237304.984375   
..         ...    ...  ...         ...          ...       ...            ...   
527       sam1     33   7A  123.200874  1165.370483  0.087026   37740.867188   
528       sam1     34  12B  123.311607  1125.358154  0.089241   35823.265625   
529       sam1     35  11A  123.322075  2128.718506  0.103856   92755.687500   
530       sam1     36   2B  123.369019   619.555359  0.105405   14698.859375   
531       sam1     37   4B  125.236382  2599.864014  0.094563  125008.953125   

     danceability  onset_rate          

In [54]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import joblib

features = [
    "bpm", "loudness", "rms", "energy",
    "danceability", "onset_rate", 
    # "dynamic_complexity", "dynamic_loudness", 
    "entropy", "crest", "centroid", "flatness", 
    "hfc", "zcr", "flux"
]

scaler = MinMaxScaler()

data[features] = scaler.fit_transform(data[features])

joblib.dump(scaler, 'scaler.pkl')

camelot = data['key'].str.extract('(?P<num>\d+)(?P<letter>[AB])')
nums = camelot['num'].astype(float)
# data['camelot_num'] = data['key'].str.extract('(\d+)').astype(int)
data['camelot_sin'] = np.sin(2 * np.pi * nums / 12)
data['camelot_cos'] = np.cos(2 * np.pi * nums / 12)
data['is_minor'] = (camelot['letter'] == 'A').astype(int)
print(data)


           mix  track  key       bpm  loudness       rms    energy  \
0    yousuke_1      0   1A  0.254512  0.046251  0.223466  0.010178   
1    yousuke_1      1   7B  0.528525  0.172499  0.361502  0.072591   
2    yousuke_1      2   5B  0.529211  0.172397  0.382146  0.072527   
3    yousuke_1      3   4A  0.495657  0.235744  0.332246  0.115704   
4    yousuke_1      4   5A  0.463121  0.253560  0.306903  0.128995   
..         ...    ...  ...       ...       ...       ...       ...   
527       sam1     33   7A  0.231969  0.073974  0.200117  0.020515   
528       sam1     34  12B  0.233076  0.071434  0.205232  0.019473   
529       sam1     35  11A  0.233181  0.135127  0.238987  0.050420   
530       sam1     36   2B  0.233650  0.039327  0.242564  0.007990   
531       sam1     37   4B  0.252326  0.165034  0.217524  0.067953   

     danceability  onset_rate                                   dynamics  ...  \
0        0.117408    0.078632   (7.961207866668701, -21.758411407470703)  ... 

In [55]:
features.append("is_minor")
features.append("camelot_sin")
features.append("camelot_cos")

for feature in features:
    data["running_" + feature] = data.groupby('mix')[feature].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    
data = data.fillna(0.0)
for feature in features:
    data["delta_" + feature] = data[feature] - data["running_" + feature]

print(data.head(20))

             mix  track  key       bpm  loudness       rms    energy  \
0      yousuke_1      0   1A  0.254512  0.046251  0.223466  0.010178   
1      yousuke_1      1   7B  0.528525  0.172499  0.361502  0.072591   
2      yousuke_1      2   5B  0.529211  0.172397  0.382146  0.072527   
3      yousuke_1      3   4A  0.495657  0.235744  0.332246  0.115704   
4      yousuke_1      4   5A  0.463121  0.253560  0.306903  0.128995   
5      yousuke_1      5   6B  0.449894  0.284995  0.300194  0.153577   
6      yousuke_1      6   5B  0.435391  0.212088  0.383257  0.098811   
7      yousuke_1      7  12B  0.372928  0.195669  0.322003  0.087614   
8      yousuke_1      8   4B  0.359815  0.134830  0.368404  0.050255   
9      yousuke_1      9   4B  0.355117  0.284015  0.350728  0.152790   
10     yousuke_1     10  11B  0.324493  0.249609  0.325631  0.126007   
11     yousuke_1     11  12A  0.297207  0.180182  0.289809  0.077469   
12     yousuke_1     12   5B  0.319813  0.235440  0.322976  0.11

In [56]:
XX, yy = [], []

for feature in features:
    data["next_" + feature] = data.groupby(["mix"])[feature].shift(-1)
data = data.groupby(["mix"]).head(-1)

print(data.head(20))

# for _, mix in data.groupby("mix"):
#     current = mix.drop(["track", "key", "dynamics"], axis=1).values
#     next = mix.drop(["track", "key", "dynamics"], axis=1).shift(-1).values
#     XX.append(current[:-1])
#     yy.append(next[:-1])
    
# XX = np.vstack(X)
# yy = np.vstack(y)
# print(X)
# print(y)

             mix  track  key       bpm  loudness       rms    energy  \
0      yousuke_1      0   1A  0.254512  0.046251  0.223466  0.010178   
1      yousuke_1      1   7B  0.528525  0.172499  0.361502  0.072591   
2      yousuke_1      2   5B  0.529211  0.172397  0.382146  0.072527   
3      yousuke_1      3   4A  0.495657  0.235744  0.332246  0.115704   
4      yousuke_1      4   5A  0.463121  0.253560  0.306903  0.128995   
5      yousuke_1      5   6B  0.449894  0.284995  0.300194  0.153577   
6      yousuke_1      6   5B  0.435391  0.212088  0.383257  0.098811   
7      yousuke_1      7  12B  0.372928  0.195669  0.322003  0.087614   
8      yousuke_1      8   4B  0.359815  0.134830  0.368404  0.050255   
9      yousuke_1      9   4B  0.355117  0.284015  0.350728  0.152790   
10     yousuke_1     10  11B  0.324493  0.249609  0.325631  0.126007   
11     yousuke_1     11  12A  0.297207  0.180182  0.289809  0.077469   
12     yousuke_1     12   5B  0.319813  0.235440  0.322976  0.11

In [73]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)

groups = data["mix"].values
# print(data)

data_clean = data.drop(["track", "key", "dynamics"], axis=1)
print(groups)

train_idx, test_idx = next(gss.split(data_clean, groups=data_clean["mix"]))

train_data = data_clean.drop("mix", axis=1).iloc[train_idx]
test_data = data_clean.drop("mix", axis=1).iloc[test_idx]

y_train = train_data.filter(like='next_')
X_train = train_data.drop(y_train.columns, axis=1)


y_test = test_data.filter(like='next_')
X_test = test_data.drop(y_test.columns, axis=1)

print(X_train)





<StringArray>
['yousuke_1', 'yousuke_1', 'yousuke_1', 'yousuke_1', 'yousuke_1', 'yousuke_1',
 'yousuke_1', 'yousuke_1', 'yousuke_1', 'yousuke_1',
 ...
      'sam1',      'sam1',      'sam1',      'sam1',      'sam1',      'sam1',
      'sam1',      'sam1',      'sam1',      'sam1']
Length: 518, dtype: str
          bpm  loudness       rms    energy  danceability  onset_rate  \
0    0.254512  0.046251  0.223466  0.010178      0.117408    0.078632   
1    0.528525  0.172499  0.361502  0.072591      0.270298    0.509277   
2    0.529211  0.172397  0.382146  0.072527      0.384466    0.652564   
3    0.495657  0.235744  0.332246  0.115704      0.290319    0.744695   
4    0.463121  0.253560  0.306903  0.128995      0.377984    0.533173   
..        ...       ...       ...       ...           ...         ...   
441  0.430924  0.255391  0.694484  0.130388      0.215103    0.435897   
442  0.434134  0.235845  0.534133  0.115778      0.246736    0.424929   
443  0.433835  0.172175  0.422165  0

In [74]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error

base_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model = MultiOutputRegressor(base_model)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')
for i, col in enumerate(y_test.columns):
    print(f"{col} Error (MAE): {round(mae[i], 4)}")

next_bpm Error (MAE): 0.076
next_loudness Error (MAE): 0.0723
next_rms Error (MAE): 0.0675
next_energy Error (MAE): 0.0511
next_danceability Error (MAE): 0.0769
next_onset_rate Error (MAE): 0.1183
next_entropy Error (MAE): 0.0633
next_crest Error (MAE): 0.0985
next_centroid Error (MAE): 0.1147
next_flatness Error (MAE): 0.0699
next_hfc Error (MAE): 0.0471
next_zcr Error (MAE): 0.1258
next_flux Error (MAE): 0.0811
next_is_minor Error (MAE): 0.4693
next_camelot_sin Error (MAE): 0.6485
next_camelot_cos Error (MAE): 0.5869


In [75]:
all_importances = np.mean([est.feature_importances_ for est in model.estimators_], axis=0)

overall_importance = pd.Series(all_importances, index=X_train.columns)
print("--- Overall Feature Importance ---")
print(overall_importance.sort_values(ascending=False))

--- Overall Feature Importance ---
rms                     0.084374
running_rms             0.060292
bpm                     0.046967
running_hfc             0.045454
running_energy          0.038299
danceability            0.028983
zcr                     0.028952
crest                   0.027934
running_flatness        0.025775
onset_rate              0.024834
running_bpm             0.024612
hfc                     0.024603
running_danceability    0.024261
songs                   0.024112
delta_camelot_sin       0.023104
position                0.022631
running_onset_rate      0.020908
running_crest           0.019650
flatness                0.019420
centroid                0.019199
running_zcr             0.019099
running_entropy         0.018152
delta_camelot_cos       0.018029
delta_rms               0.017432
delta_onset_rate        0.016897
delta_danceability      0.016844
delta_bpm               0.016580
delta_hfc               0.016565
running_centroid        0.016400
running_